# 02 - Autograd: Automatic Differentiation

**Goal:** Understand how PyTorch computes gradients automatically.

---

## Why Gradients?

From Phase 0, you learned:
- Training = adjusting weights to minimize loss
- **Gradients** tell us which direction to adjust each weight

Computing gradients by hand is tedious. PyTorch does it automatically with **autograd**.

## requires_grad: Tracking Gradients

Set `requires_grad=True` to tell PyTorch: "I want gradients for this tensor."

In [ ]:
import torch

# Regular tensor - no gradient tracking
x = torch.tensor([1.0, 2.0, 3.0])
print(f"x.requires_grad: {x.requires_grad}")  # False

# Tensor with gradient tracking
w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print(f"w.requires_grad: {w.requires_grad}")  # True

## Forward Pass → Backward Pass

When you do operations on tensors with `requires_grad=True`, PyTorch builds a **computation graph**:

```
Forward:  x → multiply by w → add b → result
Backward: gradients flow back through each operation
```

In [ ]:
# Simple example: y = w * x + b

# Input (fixed, no gradients needed)
x = torch.tensor([2.0])

# Parameters (we want to learn these)
w = torch.tensor([3.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

# Forward pass
y = w * x + b
print(f"y = w * x + b = {w.item()} * {x.item()} + {b.item()} = {y.item()}")

# y has grad_fn - it knows how it was computed
print(f"y.grad_fn: {y.grad_fn}")

In [ ]:
# Backward pass - compute gradients
y.backward()

# Now w.grad and b.grad contain the gradients
print(f"dy/dw = {w.grad}")  # Should be x = 2.0
print(f"dy/db = {b.grad}")  # Should be 1.0

# Interpretation:
# - If we increase w by 1, y increases by 2 (the value of x)
# - If we increase b by 1, y increases by 1

## Gradient = Slope

The gradient tells you the **slope** of the output with respect to each input.

```
y = w * x + b
dy/dw = x     (how much y changes when w changes)
dy/db = 1     (how much y changes when b changes)
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize: gradient is the slope
# y = w^2 (simple quadratic)

w_values = np.linspace(-3, 3, 100)
y_values = w_values ** 2

# Pick a point to show gradient
w_point = 2.0
y_point = w_point ** 2
gradient = 2 * w_point  # dy/dw = 2w

# Tangent line at that point
tangent_x = np.linspace(0.5, 3.5, 50)
tangent_y = gradient * (tangent_x - w_point) + y_point

plt.figure(figsize=(8, 5))
plt.plot(w_values, y_values, 'b-', linewidth=2, label='y = w²')
plt.plot(tangent_x, tangent_y, 'r--', linewidth=2, label=f'Tangent (slope={gradient})')
plt.scatter([w_point], [y_point], color='red', s=100, zorder=5)
plt.xlabel('w')
plt.ylabel('y')
plt.title('Gradient = Slope of the Tangent Line')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(-1, 10)
plt.show()

print(f"At w={w_point}: gradient = {gradient}")
print(f"Meaning: if w increases by 1, y increases by ~{gradient}")

## A More Realistic Example

Let's simulate one step of training:

In [ ]:
# Problem: predict y from x
# True relationship: y = 2*x + 1

# Training data
x = torch.tensor([1.0, 2.0, 3.0, 4.0])
y_true = torch.tensor([3.0, 5.0, 7.0, 9.0])  # 2*x + 1

# Our model parameters (random initialization)
w = torch.tensor([0.5], requires_grad=True)  # Should become 2.0
b = torch.tensor([0.0], requires_grad=True)  # Should become 1.0

print(f"Initial: w={w.item():.2f}, b={b.item():.2f}")
print(f"Target:  w=2.0, b=1.0")

In [ ]:
# Forward pass: make predictions
y_pred = w * x + b
print(f"Predictions: {y_pred.detach().numpy()}")
print(f"True values: {y_true.numpy()}")

# Loss: Mean Squared Error
loss = ((y_pred - y_true) ** 2).mean()
print(f"\nLoss (MSE): {loss.item():.4f}")

In [ ]:
# Backward pass: compute gradients
loss.backward()

print(f"Gradient for w: {w.grad.item():.4f}")
print(f"Gradient for b: {b.grad.item():.4f}")

# Both are negative, meaning:
# - Increasing w will DECREASE the loss (good!)
# - Increasing b will DECREASE the loss (good!)

In [ ]:
# Update parameters (one step of gradient descent)
learning_rate = 0.01

with torch.no_grad():  # Don't track this operation
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad

print(f"After 1 step: w={w.item():.4f}, b={b.item():.4f}")
print(f"Target:       w=2.0, b=1.0")
print("\nParameters moved toward the target!")

## Important: Zero Gradients!

Gradients **accumulate** by default. You must zero them before each backward pass.

In [ ]:
# Demo: gradients accumulate
w = torch.tensor([1.0], requires_grad=True)

# First backward
y = w * 2
y.backward()
print(f"After 1st backward: w.grad = {w.grad}")

# Second backward (without zeroing)
y = w * 2
y.backward()
print(f"After 2nd backward: w.grad = {w.grad}")  # Accumulated!

# Fix: zero gradients
w.grad.zero_()
y = w * 2
y.backward()
print(f"After zero + backward: w.grad = {w.grad}")  # Correct

## torch.no_grad(): Disabling Gradient Tracking

Use `torch.no_grad()` when you don't need gradients:
- During inference (prediction)
- When updating parameters
- Saves memory and computation

In [ ]:
w = torch.tensor([1.0], requires_grad=True)

# With gradient tracking (for training)
y = w * 2
print(f"With tracking - y.requires_grad: {y.requires_grad}")

# Without gradient tracking (for inference)
with torch.no_grad():
    y = w * 2
    print(f"No tracking - y.requires_grad: {y.requires_grad}")

# Your production code uses this:
# with torch.no_grad():
#     prediction = model(image)  # Fast inference, no gradient overhead

## The Training Loop Pattern

Now you understand every line of the training loop:

```python
for epoch in range(num_epochs):
    # 1. Forward pass
    predictions = model(inputs)
    loss = loss_function(predictions, targets)
    
    # 2. Backward pass
    optimizer.zero_grad()  # Clear old gradients
    loss.backward()        # Compute new gradients
    
    # 3. Update weights
    optimizer.step()       # Apply gradients
```

In [ ]:
# Full training loop example

# Data
x = torch.tensor([1.0, 2.0, 3.0, 4.0])
y_true = torch.tensor([3.0, 5.0, 7.0, 9.0])  # y = 2x + 1

# Parameters
w = torch.tensor([0.0], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

learning_rate = 0.1
losses = []

# Training loop
for epoch in range(100):
    # Forward
    y_pred = w * x + b
    loss = ((y_pred - y_true) ** 2).mean()
    losses.append(loss.item())
    
    # Backward
    loss.backward()
    
    # Update (manual gradient descent)
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    
    # Zero gradients for next iteration
    w.grad.zero_()
    b.grad.zero_()

print(f"Final: w={w.item():.4f}, b={b.item():.4f}")
print(f"Target: w=2.0, b=1.0")

In [ ]:
# Plot the loss over training
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Loss Decreasing During Training')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Loss went from {losses[0]:.4f} to {losses[-1]:.6f}")

## Summary

| Concept | What it does |
|---------|-------------|
| `requires_grad=True` | Tell PyTorch to track gradients for this tensor |
| `.backward()` | Compute gradients (backpropagation) |
| `.grad` | Access the computed gradient |
| `.zero_()` | Zero out gradients (call before each backward) |
| `torch.no_grad()` | Disable gradient tracking (for inference) |
| `.detach()` | Create a copy without gradient connection |

**Key insight:** PyTorch builds a computation graph during forward pass, then traverses it backwards to compute gradients. This is **automatic differentiation** - you don't write gradient math by hand.

**Next:** nn.Module - how to organize neural networks in PyTorch.